#PySpark Aggregations

In [0]:
#Creates the data frame
df=spark.read.table("samples.bakehouse.sales_transactions")

#displays the data frame
display(df)

In [0]:
#Code to count the amount of records
df.count()

In [0]:
df.first() #Grabs the first row of the data frame.
#df.last() #Grabs the last row of the data frame.

In [0]:
#import code
from pyspark.sql.functions import min, max

#Grabs the minimum and maximum of the quantity column
df2=df.select(min("quantity"), max("quantity"))

#displays the data frame
display(df2)

In [0]:
#import code
from pyspark.sql.functions import sum, avg

#Grabs the sum and avg of the quantity column
df2=df.select(sum("quantity"), avg("quantity"))

#displays the data frame
display(df2)

In [0]:
%sql
SELECT
  count(*), --counts the records. You need to give count a parameter greater than or equal to 1 otherwise it won't work. Use *
  min(quantity), --Grabs the minimum quantity column
  max(quantity), --Grabs the maximum and avg of the quantity column
  avg(quantity), --Grabs the avg of the quantity column
  sum(quantity) --Grabs the sum of the quantity column
FROM samples.bakehouse.sales_transactions

In [0]:
#Counts a separate aggregation. This uses a different column this time.
df2=df.groupBy("paymentMethod").count()

#displays the data frame
display(df2)

In [0]:
%sql
--Counts a separate aggregation. This uses a different column this time.
SELECT
  paymentMethod,
  count(*),
  avg(quantity)
FROM samples.bakehouse.sales_transactions
  GROUP BY paymentMethod

  --having count(*) > 1100 --Only displats records that are higher than 1100

In [0]:
#import code
from pyspark.sql.functions import col

#Applies Col to look for Payment Method them filters the records based off the values being greater than 1100.
df2=df.groupBy("paymentMethod").count().filter(col("count") > 1100)

#displays the data frame
display(df2)

In [0]:
#import code
from pyspark.sql.functions import collect_set, collect_list

#Grabs data from the payment method column and then collects the set and lists it.
df2=df.groupBy("paymentMethod").agg(collect_set("product"), collect_list("product"))

#displays the data frame
display(df2)

In [0]:
%sql
--Grabs data from the payment method column and then collects the set and lists it.
SELECT
  paymentMethod,
  collect_set(product),
  collect_list(product)
FROM samples.bakehouse.sales_transactions
  GROUP BY paymentMethod

In [0]:
#import code
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

#Defines the window spect to use in other code. This line partitions the product data then orders it by the dataframe's quantity column then shows it in descending order.
window_spec = Window.partitionBy("product").orderBy(df["quantity"].desc())

#Grabs data from the data frame and places it into a custom column.
df2=df.withColumn("row_number_over_product", row_number().over(window_spec))

#selects the product, customerID, quantity, and row_number_over_product columns
df3=df2.select("product", "customerID", "quantity", "row_number_over_product")

#displays the data frame
display(df3)

In [0]:
#filters the records above
df4=df3.filter(df3["row_number_over_product"] ==1)

#displays the data frame
display(df4)

In [0]:
#import code
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, rank, dense_rank

#Defines the window spect to use in other code. This line partitions the product data then orders it by the dataframe's quantity column then shows it in descending order.
window_spec = Window.partitionBy("product").orderBy(df["quantity"].desc())

#Grabs data from the data frame and places it into a custom column.
df2=df.withColumn("row_number_over_product", row_number().over(window_spec))\
    .withColumn("rank_over_product", rank().over(window_spec))\
    .withColumn("dense_rank_over_product", dense_rank().over(window_spec))

#selects the product, customerID, quantity, row_number_over_product, rank_over_product, and dense_rank_over_product columns
df3=df2.select(
    "product",
    "customerID",
    "quantity",
    "row_number_over_product",
    "rank_over_product",
    "dense_rank_over_product"
    )

#displays the data frame
display(df3)

In [0]:
%sql
--Defines the window spect to use in other code. This partitions the product data then orders it by the dataframe's quantity column then shows it in descending order.
SELECT
  product,
  customerID,
  quantity,
  ROW_NUMBER() OVER (PARTITION BY product ORDER BY quantity DESC) AS row_number_over_product,
  RANK() OVER (PARTITION BY product ORDER BY quantity DESC) AS rank_over_product,
  DENSE_RANK() OVER (PARTITION BY product ORDER BY quantity DESC) AS dense_rank_over_product
FROM samples.bakehouse.sales_transactions